# 📄 LaTeX & PDF Server — chạy CHUNG vLLM với OCR server

> Nhận text/markdown từ OCR → sửa chính tả → viết lại LaTeX giữ cấu trúc → export PDF bằng `xelatex` (tiếng Việt).

## ⚠️ Có 2 chế độ chạy

**Chế độ A (khuyến nghị) — dùng chung vLLM với OCR server:**
Nếu đã chạy `OCR_API_Server.ipynb` với Qwen3-VL-35B-A3B thì GPU không đủ cho instance thứ 2.
→ **BỎ QUA Cell 1, Cell 3**. Chỉ chạy Cell 2 (TeX Live) + Cell 4 (pip) + Cell 5 → 7.
LaTeX API sẽ gọi HTTP sang vLLM của OCR server (`localhost:8008`).

**Chế độ B — chạy vLLM riêng (chỉ khi có GPU thứ 2 hoặc không chạy OCR cùng lúc):**
Chạy đủ Cell 1 → 7.

## Endpoints
- `POST /latex` — body `{"text": "..."}` → trả LaTeX
- `POST /pdf` — body `{"text": "..."}` → trả file PDF
- `POST /compile` — body `{"latex": "..."}` → trả PDF (bỏ qua LLM)
- `POST /latex/stream` — streaming SSE
- `GET  /health`


## Cell 1 — Cài vLLM

In [ ]:
# ==================== Cài vLLM ====================
!pip uninstall -q vllm -y
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q vllm --extra-index-url https://wheels.vllm.ai/nightly
print("✅ Cài vLLM xong!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0

## Cell 2 — Cài TeX Live (xelatex) + font tiếng Việt
> Mất ~3–5 phút lần đầu. TeX Live đầy đủ để có `xelatex`, `polyglossia`, `fontspec`.

In [ ]:
# ==================== Cài TeX Live (xelatex) ====================
import subprocess, shutil

if shutil.which("xelatex") is None:
    print("⌛ Đang cài texlive-xetex (có thể mất vài phút) ...")
    subprocess.run(
        "apt-get update -qq && "
        "apt-get install -y -q texlive-xetex texlive-lang-other "
        "texlive-latex-extra texlive-fonts-recommended "
        "fonts-dejavu fonts-liberation lmodern latexmk",
        shell=True, check=True
    )
    print("✅ Cài xong texlive-xetex")
else:
    print("✅ xelatex đã có sẵn")

print("xelatex :", shutil.which("xelatex"))
print("latexmk :", shutil.which("latexmk"))

⌛ Đang cài texlive-xetex (có thể mất vài phút) ...
✅ Cài xong texlive-xetex
xelatex : /usr/bin/xelatex
latexmk : /usr/bin/latexmk


## Cell 3 — Khởi động vLLM server

In [42]:
import subprocess, threading, time
from IPython.display import clear_output

def start_vllm():
    cmd = [
        "vllm", "serve", "Qwen/Qwen3.6-35B-A3B",
        "--port", "8888",
        "--gpu-memory-utilization", "0.9",
        "--kv-cache-dtype", "fp8_e5m2",
        "--dtype", "bfloat16",
        "--max-model-len", "32768",
        "--max-num-seqs", "64",
        "--max-num-batched-tokens", "16384",
        "--enable-prefix-caching",
        "--enable-chunked-prefill",
        "--trust-remote-code",
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')

thread = threading.Thread(target=start_vllm, daemon=True)
thread.start()
print("⌛ Đang load model Qwen3.6-35B-A3B... (~60-90s)")
time.sleep(75)
clear_output(wait=True)
print("✅ vLLM Server đang chạy tại localhost:8888")

✅ vLLM Server đang chạy tại localhost:8080


## Cell 4 — Cài thư viện API

In [ ]:
!pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles sse-starlette requests
print("✅ Cài xong dependencies API")

✅ Cài xong dependencies API


## Cell 5 — Định nghĩa FastAPI app (LaTeX + PDF)

In [ ]:
import os, re, time, json, uuid, tempfile, subprocess, shutil, asyncio
import requests as req_sync
from pathlib import Path
from typing import Optional

from fastapi import FastAPI, HTTPException, Body
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
from sse_starlette.sse import EventSourceResponse

# ─── Config ────────────────────────────────────────────────────────────
# 👉 Dùng chung vLLM với OCR server. KHÔNG chạy Cell 1-3.
#    Sửa VLLM_PORT cho khớp port vLLM của OCR server đang chạy.
try:
    VLLM_PORT
except NameError:
    VLLM_PORT = 8000
try:
    MODEL_NAME
except NameError:
    MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"
VLLM_URL = f"http://localhost:{VLLM_PORT}/v1/chat/completions"

DEBUG_DIR = Path("/content/debug_latex")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

# ─── System prompt ─────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Bạn là biên tập viên tiếng Việt kiêm chuyên gia LaTeX. Nhiệm vụ:\n"
    "1. Nhận văn bản đã OCR (markdown hoặc plain text) — có thể sai chính tả, "
    "thiếu dấu, lộn dòng, ký tự nhiễu.\n"
    "2. Sửa chính tả, dấu câu, lỗi OCR rõ ràng. TUYỆT ĐỐI KHÔNG thêm nội dung không có trong bản gốc, "
    "không tóm tắt, không dịch, không bịa số liệu.\n"
    "3. Viết lại dưới dạng LaTeX, GIỮ NGUYÊN cấu trúc: tiêu đề → \\section/\\subsection, "
    "danh sách → itemize/enumerate, bảng → tabular, in đậm → \\textbf, in nghiêng → \\textit, "
    "công thức toán → \\(...\\) hoặc \\[...\\].\n"
    "4. Escape đúng các ký tự đặc biệt LaTeX: % $ & # _ { } ~ ^ \\ .\n"
    "5. CHỈ trả về phần NỘI DUNG LaTeX (những gì nằm giữa \\begin{document} và \\end{document}), "
    "KHÔNG kèm \\documentclass, KHÔNG kèm preamble, KHÔNG kèm ``` hoặc giải thích.\n"
    "6. KHÔNG viết <think>, không viết suy nghĩ. Trả lời trực tiếp bằng LaTeX."
)

# ─── Preamble (xelatex + fontspec + DejaVu cho tiếng Việt) ──────────────
LATEX_PREAMBLE = r"""\documentclass[12pt,a4paper]{article}
\usepackage{fontspec}
\defaultfontfeatures{Ligatures=TeX}
\setmainfont{DejaVu Serif}
\setsansfont{DejaVu Sans}
\setmonofont{DejaVu Sans Mono}
\usepackage[margin=2.2cm]{geometry}
\usepackage{amsmath,amssymb,amsfonts}
\usepackage{graphicx}
\usepackage{array,booktabs,longtable}
\usepackage{hyperref}
\usepackage{enumitem}
\usepackage{parskip}
\setlength{\parindent}{0pt}
\begin{document}
"""
LATEX_POSTAMBLE = "\n\\end{document}\n"

# ─── Helpers ───────────────────────────────────────────────────────────
def call_qwen(user_text: str, stream: bool = False, max_tokens: int = 8192):
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_text},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.1,
        "top_p": 0.9,
        "stream": stream,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    return req_sync.post(VLLM_URL, json=payload, timeout=600, stream=stream)


_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_FENCE_OPEN_RE  = re.compile(r"^\s*```(?:latex|tex)?\s*\n", re.IGNORECASE)
_FENCE_CLOSE_RE = re.compile(r"\n?\s*```\s*$")
_DOC_RE   = re.compile(r"\\begin\{document\}(.*?)\\end\{document\}", re.DOTALL)

def strip_wrapping(latex: str) -> str:
    if not latex:
        return ""
    latex = _THINK_RE.sub("", latex).strip()
    latex = _FENCE_OPEN_RE.sub("", latex)
    latex = _FENCE_CLOSE_RE.sub("", latex).strip()
    m = _DOC_RE.search(latex)
    if m:
        latex = m.group(1).strip()
    return latex

def build_full_tex(body: str) -> str:
    return LATEX_PREAMBLE + body.strip() + LATEX_POSTAMBLE

def compile_pdf(tex_source: str, engine: str = "xelatex", debug_id: str = None):
    with tempfile.TemporaryDirectory() as td:
        td_path = Path(td)
        tex_file = td_path / "doc.tex"
        tex_file.write_text(tex_source, encoding="utf-8")
        cmd = [engine, "-interaction=nonstopmode", "-halt-on-error",
               "-output-directory", str(td_path), str(tex_file)]
        last = None
        for _ in range(2):
            last = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
            if last.returncode != 0:
                break
        pdf_file = td_path / "doc.pdf"
        log_file = td_path / "doc.log"
        log_txt = ""
        if log_file.exists():
            try: log_txt = log_file.read_text(encoding="utf-8", errors="ignore")
            except: pass
        if debug_id:
            try: (DEBUG_DIR / f"{debug_id}.log").write_text(log_txt, encoding="utf-8")
            except: pass
        if not pdf_file.exists():
            raise HTTPException(
                status_code=500,
                detail=f"{engine} failed.\nSTDOUT:\n{(last.stdout if last else '')[-2000:]}\n\nLOG:\n{log_txt[-4000:]}"
            )
        return pdf_file.read_bytes(), log_txt[-2000:]


def save_debug(debug_id: str, raw: str, body: str, tex: str):
    try:
        (DEBUG_DIR / f"{debug_id}.raw.txt").write_text(raw or "", encoding="utf-8")
        (DEBUG_DIR / f"{debug_id}.body.tex").write_text(body or "", encoding="utf-8")
        (DEBUG_DIR / f"{debug_id}.full.tex").write_text(tex or "", encoding="utf-8")
    except Exception as e:
        print("save_debug err:", e)


# ─── App ────────────────────────────────────────────────────────────────
app = FastAPI(title="LaTeX & PDF Server", version="1.2.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)

class TextIn(BaseModel):
    text: str
    max_tokens: Optional[int] = 8192
    engine: Optional[str] = "xelatex"

class LaTeXIn(BaseModel):
    latex: str
    engine: Optional[str] = "xelatex"
    full_document: Optional[bool] = False


@app.get("/health")
def health():
    detail = None
    try:
        r = req_sync.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=5)
        models = [m["id"] for m in r.json().get("data", [])]
        vllm_ready = True
    except Exception as e:
        models, vllm_ready = [], False
        detail = str(e)
    out = {
        "status": "ok",
        "vllm": "ready" if vllm_ready else "loading",
        "vllm_port": VLLM_PORT,
        "model_name": MODEL_NAME,
        "models": models,
        "xelatex": shutil.which("xelatex") is not None,
        "lualatex": shutil.which("lualatex") is not None,
    }
    if detail: out["detail"] = detail
    return out


@app.post("/latex")
def to_latex(inp: TextIn):
    if not inp.text.strip():
        raise HTTPException(400, "text rỗng")
    t0 = time.time()
    debug_id = uuid.uuid4().hex[:8]
    r = call_qwen(inp.text, stream=False, max_tokens=inp.max_tokens or 8192)
    if r.status_code != 200:
        raise HTTPException(502, f"vLLM error {r.status_code}: {r.text[:500]}")
    raw  = r.json()["choices"][0]["message"]["content"] or ""
    body = strip_wrapping(raw)
    tex  = build_full_tex(body)
    save_debug(debug_id, raw, body, tex)
    return {
        "debug_id": debug_id,
        "raw_len": len(raw),
        "body_len": len(body),
        "raw_head": raw[:400],
        "latex_body": body,
        "full_document": tex,
        "time_s": round(time.time() - t0, 2),
    }


@app.post("/pdf")
def to_pdf(inp: TextIn):
    if not inp.text.strip():
        raise HTTPException(400, "text rỗng")
    debug_id = uuid.uuid4().hex[:8]
    r = call_qwen(inp.text, stream=False, max_tokens=inp.max_tokens or 8192)
    if r.status_code != 200:
        raise HTTPException(502, f"vLLM error {r.status_code}: {r.text[:500]}")
    raw  = r.json()["choices"][0]["message"]["content"] or ""
    body = strip_wrapping(raw)
    if not body.strip():
        save_debug(debug_id, raw, body, "")
        raise HTTPException(
            status_code=422,
            detail=f"LLM trả về rỗng sau khi strip. debug_id={debug_id}. raw_head={raw[:400]!r}"
        )
    tex_src = build_full_tex(body)
    save_debug(debug_id, raw, body, tex_src)
    pdf, _ = compile_pdf(tex_src, engine=(inp.engine or "xelatex"), debug_id=debug_id)
    fname = f"document_{debug_id}.pdf"
    return Response(
        content=pdf, media_type="application/pdf",
        headers={"Content-Disposition": f'attachment; filename="{fname}"',
                 "X-Debug-Id": debug_id}
    )


@app.post("/compile")
def compile_only(inp: LaTeXIn):
    debug_id = uuid.uuid4().hex[:8]
    tex_src = inp.latex if inp.full_document else build_full_tex(strip_wrapping(inp.latex))
    save_debug(debug_id, inp.latex, strip_wrapping(inp.latex), tex_src)
    pdf, _ = compile_pdf(tex_src, engine=(inp.engine or "xelatex"), debug_id=debug_id)
    fname = f"document_{debug_id}.pdf"
    return Response(
        content=pdf, media_type="application/pdf",
        headers={"Content-Disposition": f'attachment; filename="{fname}"',
                 "X-Debug-Id": debug_id}
    )


@app.get("/debug/{debug_id}")
def debug_get(debug_id: str):
    out = {}
    for suffix in ("raw.txt", "body.tex", "full.tex", "log"):
        p = DEBUG_DIR / f"{debug_id}.{suffix}"
        if p.exists():
            try: out[suffix] = p.read_text(encoding="utf-8", errors="ignore")
            except Exception as e: out[suffix] = f"<read err: {e}>"
        else:
            out[suffix] = None
    if all(v is None for v in out.values()):
        raise HTTPException(404, f"debug_id {debug_id} not found")
    return out


@app.post("/latex/stream")
async def latex_stream(inp: TextIn):
    async def gen():
        if not inp.text.strip():
            yield {"data": json.dumps({"type": "error", "detail": "text rỗng"})}
            return
        loop = asyncio.get_event_loop()
        r = await loop.run_in_executor(None, lambda: call_qwen(inp.text, stream=True, max_tokens=inp.max_tokens or 8192))
        if r.status_code != 200:
            yield {"data": json.dumps({"type": "error", "detail": f"vLLM {r.status_code}"})}
            return
        yield {"data": json.dumps({"type": "start"})}
        for raw in r.iter_lines(decode_unicode=True):
            if not raw or not raw.startswith("data:"): continue
            chunk = raw[5:].strip()
            if chunk == "[DONE]":
                break
            try:
                obj = json.loads(chunk)
                delta = obj["choices"][0]["delta"].get("content", "")
                if delta:
                    yield {"data": json.dumps({"type": "delta", "text": delta})}
            except Exception:
                continue
        yield {"data": json.dumps({"type": "done"})}

    return EventSourceResponse(gen())


print("✅ FastAPI app (v1.2) đã định nghĩa.")
print(f"   → vLLM: {VLLM_URL}  | model: {MODEL_NAME}")
print(f"   → Debug dir: {DEBUG_DIR}")
print("   Endpoints: /health  /latex  /pdf  /compile  /debug/{id}  /latex/stream")


## Cell 6 — Khởi động FastAPI + tunnel ra ngoài

In [36]:
import threading, time, os, subprocess, re
import requests as req_sync
import uvicorn

API_PORT  = 8901
SUBDOMAIN = "vks-latex-hvks"
FIXED_URL = f"https://{SUBDOMAIN}.loca.lt"

print("[1/5] start uvicorn ...", flush=True)

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(6)

print("[2/5] check /health ...", flush=True)
try:
    r = req_sync.get(f"http://localhost:{API_PORT}/health", timeout=5)
    print("      status:", r.status_code, "body:", r.text[:200], flush=True)
except Exception as e:
    print("      ⚠️ health check failed:", e, flush=True)

print("[3/5] install localtunnel if missing ...", flush=True)
if subprocess.run("which lt", shell=True, capture_output=True).returncode != 0:
    subprocess.run("npm install -g localtunnel -q", shell=True, timeout=120)
print("      done", flush=True)

print(f"[4/5] start tunnel với subdomain cố định: {SUBDOMAIN} ...", flush=True)

def _try_tunnel():
    subprocess.run("pkill -9 -f 'lt --port 8901' || true", shell=True, timeout=5)
    time.sleep(2)
    try: os.remove("tunnel_latex.log")
    except: pass
    get_ipython().system_raw(
        f"lt --port {API_PORT} --subdomain {SUBDOMAIN} > tunnel_latex.log 2>&1 &"
    )
    for _ in range(40):
        time.sleep(1)
        try:
            log = open("tunnel_latex.log").read()
            m = re.search(r"https://[\w-]+\.loca\.lt", log)
            if m: return m.group(0), log
        except FileNotFoundError:
            pass
    try: log = open("tunnel_latex.log").read()
    except: log = ""
    return None, log

clean_url = None
last_log = ""
for attempt in range(1, 4):
    print(f"      attempt {attempt}/3 ...", flush=True)
    url, last_log = _try_tunnel()
    if url == FIXED_URL:
        clean_url = url
        break
    if url:
        print(f"      ⚠️ nhận {url} thay vì {FIXED_URL}, retry ...", flush=True)
    else:
        print(f"      ⚠️ chưa nhận URL, retry ...", flush=True)

print("[5/5] kết quả", flush=True)
if clean_url:
    print(f"\n✅ URL CỐ ĐỊNH: {clean_url}")
    try:
        pw = req_sync.get("https://loca.lt/mytunnelpassword", timeout=10).text.strip()
        print(f"🔑 Tunnel password: {pw}")
    except: pass
    print(f"\n📋 Endpoints:")
    print(f"  GET  {clean_url}/health")
    print(f"  POST {clean_url}/latex         (JSON: {{text: ...}})")
    print(f"  POST {clean_url}/pdf           (JSON: {{text: ...}} → PDF)")
    print(f"  POST {clean_url}/compile       (JSON: {{latex: ...}} → PDF)")
    print(f"  POST {clean_url}/latex/stream  (SSE)")
else:
    print(f"❌ Không chiếm được subdomain '{SUBDOMAIN}' sau 3 lần thử.")
    print("Log cuối của localtunnel:"); print(last_log)
    print(f"\n→ Đổi biến SUBDOMAIN trong cell này rồi chạy lại.")

[1/5] start uvicorn ...
[2/5] check /health ...
      status: 200 body: {"status":"ok","vllm":"loading","vllm_port":8011,"model_name":"Qwen/Qwen3.6-35B-A3B","models":[],"xelatex":true,"lualatex":true}
[3/5] install localtunnel if missing ...
      done
[4/5] start tunnel với subdomain cố định: vks-latex-hvks ...
      attempt 1/3 ...
[5/5] kết quả

✅ URL CỐ ĐỊNH: https://vks-latex-hvks.loca.lt
🔑 Tunnel password: 34.12.144.182

📋 Endpoints:
  GET  https://vks-latex-hvks.loca.lt/health
  POST https://vks-latex-hvks.loca.lt/latex         (JSON: {text: ...})
  POST https://vks-latex-hvks.loca.lt/pdf           (JSON: {text: ...} → PDF)
  POST https://vks-latex-hvks.loca.lt/compile       (JSON: {latex: ...} → PDF)
  POST https://vks-latex-hvks.loca.lt/latex/stream  (SSE)


## Cell 6b — Keep-alive (giữ Colab + tunnel sống)

In [37]:
# _COLAB_KEEPALIVE_MARKER_
from IPython.display import display, Javascript
import threading, time, requests as _ka_req

display(Javascript(r"""
(function(){
  if (window.__colabKeepAliveLatex) return;
  window.__colabKeepAliveLatex = true;
  function tick(){
    try {
      document.querySelectorAll('colab-connect-button').forEach(b => {
        const s = b.shadowRoot;
        if (s) { const btn = s.querySelector('#connect'); if (btn) btn.click(); }
      });
      document.querySelectorAll('paper-icon-button[icon="notification:sync"]').forEach(b => b.click());
      document.dispatchEvent(new MouseEvent('mousemove', {clientX:1, clientY:1, bubbles:true}));
      document.dispatchEvent(new KeyboardEvent('keydown', {key:'Shift', bubbles:true}));
    } catch(e){}
  }
  setInterval(tick, 45*1000);
  console.log('[keepalive-latex] JS registered');
})();
"""))

_KA_STOP = threading.Event()
def _ka_loop():
    tunnel = None
    try: tunnel = clean_url  # noqa
    except NameError: pass
    while not _KA_STOP.is_set():
        try: _ka_req.get(f'http://localhost:{API_PORT}/health', timeout=5)
        except: pass
        if tunnel:
            try:
                _ka_req.get(tunnel + '/health',
                            headers={'bypass-tunnel-reminder': 'true'},
                            timeout=10)
            except: pass
        _KA_STOP.wait(60)

if globals().get('_ka_thread') and _ka_thread.is_alive():
    _KA_STOP.set(); _ka_thread.join(timeout=2); _KA_STOP = threading.Event()
_ka_thread = threading.Thread(target=_ka_loop, daemon=True, name='latex-keepalive')
_ka_thread.start()
print("✅ Keep-alive bật (JS 45s + self-ping 60s)")

<IPython.core.display.Javascript object>

✅ Keep-alive bật (JS 45s + self-ping 60s)


In [41]:
import requests
try:
    r = requests.get("http://localhost:8008/v1/models", timeout=5)
    print("STATUS:", r.status_code)
    print("BODY:", r.text[:500])
except Exception as e:
    print("DEAD:", type(e).__name__, e)


STATUS: 404
BODY: {"detail":"Not Found"}
